# 4. Ethane: Symmetry in Calculations

Ethane can **rotate** freely along the C–C bond. The lowest energy **conformation** is the *anti* form and the highest energy is the *syn* form. It is easy to get *GAMESS* to optimize the *anti* structure, the software will **seek** the lowest energy state. How do we get the *syn* form? We could do this by **enforcing** symmetry.

## Learning Goals

This exercise will hopefully accomplish the following **goals** while learning new **skills**:

1.	We will manually **construct** *Z-matrices* for larger molecules and identify an internal coordinate that defines the torsion angle between the two methyl groups of ethane.
2.	We will use the **symmetry** of the *anti* and *syn* conformations to **lock** their conformations.
3.	We will determine optimized **structure** and the **energy** difference between the *syn* and *anti*-conformations and **compare** our results to literature values.

:::{note} Mac users: after a restart
macOS resets the shared-memory limits every time you restart. If a *GAMESS* job stops with a fatal error and the `.log` file shows `shmget returned an error`, set the limits again (see [Raise shared-memory limits](#shared-memory-limits) in [](1D-CompileInstruction2024.md)).
:::

### Settings

Run this cell first. It loads the book's helper module, `gamess_tools.py`, which finds *GAMESS* using the location you set in `gamess_settings.py` on the [setup page](1_Setup.ipynb). All the jobs in this notebook write their `.log` files into the `runs/` folder next to the notebook.

In [1]:
# ===== SETTINGS =========================================================
# Your GAMESS location is set ONCE, in gamess_settings.py (next to this
# notebook). Edit that file, save it, and restart the kernel.
from pathlib import Path
import pandas as pd
import gamess_tools as gt

INPUTS = Path("inputs")     # input files that come with the book
RUNS = Path("runs")         # where log files and new input files are written
RUNS.mkdir(exist_ok=True)

HARTREE_TO_KJMOL = 2625.5   # 1 hartree = 2625.5 kJ/mol
# ========================================================================

## Point Groups in Ethane

Each of the extreme cases in ethane has **symmetry**. They have **different** symmetry and so we can use that to our advantage. The symmetry of the *syn* conformation is *D<sub>3h</sub>* and for the *anti* it is *D<sub>3d</sub>*. By declaring the **symmetry group**, the optimization will start with that symmetry and end with that symmetry. This will enforce the *anti* conformation for us. Additionally, by symmetry, each molecule can be reduced to just two atoms for the purposes of the calculation. Using symmetry when possible will greatly **reduce the cost** of calculation.

## *Z-Matrix* for Ethane

We can start with a *Z-matrix* for ethane. If the structure **described** fits the symmetry group declared, then *GAMESS* will **align** it appropriately in the *xyz* coordinate system and **identify** the symmetry unique atoms. Take note of how the two methyl groups are set up so that they **refer** to their own atoms as much as possible and there is only one coordinate that describes the bond **rotation** (line 16). Below is the initial input file for the *anti* conformation with *D<sub>3d</sub>* symmetry.

```{literalinclude} inputs/Ethane_anti_AM1_Z.inp
:label: ethane_anti_am1_z-inp
:caption: inputs/Ethane_anti_AM1_Z.inp
:linenos: True
:emphasize-lines: 16
```

In [2]:
anti_am1_log = gt.run_gamess(INPUTS / "Ethane_anti_AM1_Z.inp")
print(f"AM1 energy: {gt.final_energy(anti_am1_log):.10f} hartree\n")
gt.excerpt(anti_am1_log, "COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)", after=5)

Ethane_anti_AM1_Z: finished normally (Ethane_anti_AM1_Z.log)
AM1 energy: -12.4631119056 hartree

 1452   COORDINATES OF SYMMETRY UNIQUE ATOMS (ANGS)
 1453     ATOM   CHARGE       X              Y              Z
 1454   ------------------------------------------------------------
 1455   C           6.0   0.0000000000   0.0000000000  -0.7501162007
 1456   H           1.0   0.0000000000   1.0448521625   1.1453263085
 1457   COORDINATES OF ALL ATOMS ARE (ANGS)


## *anti*-Ethane

The result file produced the symmetry unique coordinates optimized at the AM1 level (printed above). I used those to **construct** an input file for the HF/6-311G(d,p)++ optimization. That file is reproduced below. The `IZMAT` matrix was **constructed** using the `ENCODED Z MATRIX` section of the AM1 result file, which *GAMESS* writes when it converts our *Z-matrix* into internal coordinates.

In [3]:
gt.excerpt(anti_am1_log, "ENCODED Z MATRIX", after=19, occurrence=0)

  448     --- ENCODED Z MATRIX ---
  449   COORD  TYPE   I   J   K   L   M   N
  450      1      1    1   2
  451      2      1    8   1
  452      3      2    8   1   2
  453      4      1    4   1
  454      5      2    4   1   2
  455      6      3    4   1   2   8
  456      7      1    3   1
  457      8      2    3   1   2
  458      9      3    3   1   2   8
  459     10      1    5   2
  460     11      2    5   2   1
  461     12      3    5   2   1   8
  462     13      1    6   2
  463     14      2    6   2   1
  464     15      3    6   2   1   5
  465     16      1    7   2
  466     17      2    7   2   1
  467     18      3    7   2   1   5


Each row is one internal coordinate: TYPE 1 is a stretch between atoms I and J, TYPE 2 a bend I–J–K and TYPE 3 a torsion I–J–K–L. Written out as `type,I,J,...` groups, these rows become the `IZMAT` list of the input file below.

```{literalinclude} inputs/Ethane_anti_6311_U.inp
:label: ethane_anti_6311_u-inp
:caption: inputs/Ethane_anti_6311_U.inp
:linenos: True
:emphasize-lines: 20
```

```{note}
I altered one of the IZMAT coordinates to correspond to a H–C–H bond angle. In the last line, I changed the `2,7,2,1` (an H–C–C angle) to `2,7,2,6` (an H–C–H angle.)
```

The file in `inputs/` is our template. *Python* copies it and puts the AM1 unique coordinates from our own run into the `$DATA` group (`gt.new_input()`, as in [chapter 2](2_SmallMolecules.ipynb)). Then we run the job.

In [4]:
anti_inp = gt.new_input(INPUTS / "Ethane_anti_6311_U.inp", RUNS / "Ethane_anti_6311_U.inp",
                        atoms=gt.final_geometry(anti_am1_log, unique=True))
anti_log = gt.run_gamess(anti_inp)

Ethane_anti_6311_U: finished normally (Ethane_anti_6311_U.log)


We get an **energy** from this calculation and a **geometry** that we can read from the `INTERNAL COORDINATES` section of the result file. This section is available because we set `NZVAR=18` and provided the `IZMAT` list. `gt.internal_coordinates()` reads the last copy of that table.

In [5]:
for kind, atoms, value in gt.internal_coordinates(anti_log):
    unit = "Å" if kind == "STRETCH" else "°"
    print(f"{kind:8s} {str(atoms):14s} {value:12.4f} {unit}")

STRETCH  (1, 2)               1.5272 Å
STRETCH  (8, 1)               1.0862 Å
BEND     (8, 1, 2)          111.1975 °
STRETCH  (4, 1)               1.0862 Å
BEND     (4, 1, 2)          111.1975 °
TORSION  (4, 1, 2, 8)      -120.0000 °
STRETCH  (3, 1)               1.0862 Å
BEND     (3, 1, 2)          111.1975 °
TORSION  (3, 1, 2, 8)       120.0000 °
STRETCH  (5, 2)               1.0862 Å
BEND     (5, 2, 1)          111.1975 °
TORSION  (5, 2, 1, 8)       180.0000 °
STRETCH  (6, 2)               1.0862 Å
BEND     (6, 2, 1)          111.1975 °
TORSION  (6, 2, 1, 5)      -120.0000 °
STRETCH  (7, 2)               1.0862 Å
BEND     (7, 2, 6)          107.6911 °
TORSION  (7, 2, 1, 5)       120.0000 °


Here is a summary of the calculated **data** at the HF/6-311G(d,p)++ level. Note that the internal coordinates chosen by *GAMESS* are not in the same pattern as the *Z-matrix*. We can manually **choose** these coordinates if we wish, but the number of coordinates must match the number of degrees of freedom (3*n*–6) and the coordinates must be able to be used to **describe** the position of all atoms.

The small function below measures the four values we want from the final geometry, whatever order *GAMESS* put the atoms in: it finds the two carbons, the three hydrogens closest to the first carbon, and then uses `gt.distance()` and `gt.angle()`.

In [6]:
def ethane_summary(log):
    # C-C and C-H lengths, H-C-C and H-C-H angles, and energy of an ethane run
    atoms = gt.final_geometry(log)
    c1, c2 = [n for n, atom in enumerate(atoms, start=1) if atom[0] == "C"]
    hydrogens = [n for n, atom in enumerate(atoms, start=1) if atom[0] == "H"]
    h1, h2, h3 = sorted(hydrogens, key=lambda h: gt.distance(atoms, c1, h))[:3]
    return {"C–C length (Å)": gt.distance(atoms, c1, c2),
            "C–H length (Å)": gt.distance(atoms, c1, h1),
            "H–C–C angle (°)": gt.angle(atoms, h1, c1, c2),
            "H–C–H angle (°)": gt.angle(atoms, h1, c1, h2),
            "Energy (Ha)": gt.nserch_energies(log)[-1][1]}

def show(value):
    # total energies (large and negative) with 10 decimals, everything else with 3
    return f"{value:.10f}" if value < -10 else f"{value:.3f}"

anti = ethane_summary(anti_log)
for name, value in anti.items():
    print(f"{name:16s} {show(value):>15s}")

C–C length (Å)             1.527
C–H length (Å)             1.086
H–C–C angle (°)          111.198
H–C–H angle (°)          107.691
Energy (Ha)       -79.2520118108


The C–C bond is 1.527 Å, the C–H bonds are 1.086 Å, the H–C–C angle is 111.2&deg; and the H–C–H angle is 107.7&deg;.

The absolute QM energy value is just a number until we compare it to a similar structure. In this case we will **compare** it to a very similar structure, the *syn* conformation. This is probably the best case scenario for calculating **relative** energies between two situations.

## *syn*-Ethane

The *syn* structure has *D<sub>3h</sub>* symmetry and the *Z-matrix* is **almost** identical. We just had to **change** the one coordinate that defined the **torsion** angle between the two methyl groups. This is the 6,1,2,3 torsion angle (line 6 of the *Z-matrix*, line 16 of the input file). I just need to change the angle to zero degrees and the symmetry to *D<sub>3h</sub>* (line 9) and **repeat** all the above. First we altered the *anti* input file and submitted the job.

```{literalinclude} inputs/Ethane_syn_AM1_Z.inp
:label: ethane_syn_am1_z-inp
:caption: inputs/Ethane_syn_AM1_Z.inp
:linenos: True
:emphasize-lines: 9, 16
```

In [7]:
syn_am1_log = gt.run_gamess(INPUTS / "Ethane_syn_AM1_Z.inp")
print(f"AM1 energy: {gt.final_energy(syn_am1_log):.10f} hartree\n")

for name, log in [("anti (D3d)", anti_am1_log), ("syn (D3h)", syn_am1_log)]:
    print(f"Unique atoms, {name}:")
    print(gt.atom_lines(gt.final_geometry(log, unique=True)))

Ethane_syn_AM1_Z: finished normally (Ethane_syn_AM1_Z.log)
AM1 energy: -12.4611212062 hartree

Unique atoms, anti (D3d):
 C        6.0    0.0000000000    0.0000000000   -0.7501162007
 H        1.0    0.0000000000    1.0448521625    1.1453263085
Unique atoms, syn (D3h):
 C        6.0    0.0000000000   -0.0000000000   -0.7517425284
 H        1.0    1.0424482146   -0.0000000000    1.1528384766


Then we **used** the unique coordinates generated to create the HF/6-311G(d,p)++ input file. Observe that the unique coordinates have been **aligned** by *GAMESS* into a **different** plane. In the *D<sub>3d</sub>* *anti* symmetry, the *yz* plane was used for the unique atoms, and in the *D<sub>3h</sub>* *syn* case, the *xz* plane was chosen. The *x*-axis is the perpendicular 2-fold axis of symmetry (that defines the *D* rather than *C* point groups). This places the primary *&sigma;<sub>d</sub>*-plane in the *yz* plane for the *anti* conformation and the *&sigma;<sub>v</sub>*-plane in the *xz* plane for the *syn* conformation.

I had tried to just change the symmetry to *D<sub>3h</sub>* and use the **previous** unique *D<sub>3d</sub>* coordinates and the symmetry **failed**. We can repeat that mistake on purpose and see how it fails.

In [8]:
wrong_inp = gt.new_input(INPUTS / "Ethane_syn_6311_U.inp", RUNS / "Ethane_syn_6311_U_wrong.inp",
                         atoms=gt.final_geometry(anti_am1_log, unique=True))   # anti coordinates!
wrong_log = gt.run_gamess(wrong_inp)
gt.excerpt(wrong_log, "YOU MUST INPUT", after=1)

Ethane_syn_6311_U_wrong: FAILED, see /Users/blink/github/UsingGAMESSv2/runs/Ethane_syn_6311_U_wrong.log
  526   YOU MUST INPUT 3N-6 INTERNAL COORDS WHERE N = 14
  527   YOU HAVE PROVIDED     18


*GAMESS* applied the *D<sub>3h</sub>* symmetry operations to a hydrogen that was not on a mirror plane of that point group, and the copies landed in new places: it built a "molecule" with 14 atoms (2 carbons and 12 hydrogens) instead of 8. A 14-atom molecule needs 3(14)–6 = 36 internal coordinates, not the 18 we gave it, so the job stopped. The unique coordinates must come from a run that used the **same** point group.

Below is the HF/6-311G(d,p)++ input file. As before, *Python* fills in the AM1 unique coordinates from the *syn* run.

```{literalinclude} inputs/Ethane_syn_6311_U.inp
:label: ethane_syn_6311_u-inp
:caption: inputs/Ethane_syn_6311_U.inp
:linenos: True
:emphasize-lines: 20
```

In [9]:
syn_inp = gt.new_input(INPUTS / "Ethane_syn_6311_U.inp", RUNS / "Ethane_syn_6311_U.inp",
                       atoms=gt.final_geometry(syn_am1_log, unique=True))
syn_log = gt.run_gamess(syn_inp)
syn = ethane_summary(syn_log)

Ethane_syn_6311_U: finished normally (Ethane_syn_6311_U.log)


## Discussion

Here is the **data** gathered together so that we can explore the **differences** in bond lengths, angles and molecular energy. The last row is the energy relative to the *anti* conformer, in kJ/mol.

In [10]:
table = pd.DataFrame({"syn conformer": syn, "anti conformer": anti})
table.loc["ΔE (kJ/mol)"] = (table.loc["Energy (Ha)"] - anti["Energy (Ha)"]) * HARTREE_TO_KJMOL
table.apply(lambda column: column.map(show))

,syn conformer,anti conformer
C–C length (Å),1.541,1.527
C–H length (Å),1.085,1.086
H–C–C angle (°),111.641,111.198
H–C–H angle (°),107.217,107.691
Energy (Ha),-79.2471515134,-79.2520118108
ΔE (kJ/mol),12.761,0.000


The C–C bond is **shorter** in the *anti* conformer (1.527 Å vs 1.541 Å), as expected. And the H–C–H angle was **larger** (107.7&deg; vs 107.2&deg;), as expected. The *syn* conformer is 12.8 kJ/mol higher in energy.

## The Controversy

The **literature** value for the ethane rotational energy barrier is 12.1 kJ/mol [@Hirota1979]. We were pretty close with this low level of theory. There has been lots of **recent** work in this subject. Despite its simplicity, or because of it, it has been a focus for theoretical investigations. It started with an argument that molecular orbital interactions **stabilizing** the *anti* conformation were the **more important** contributor to the energy difference compared to electronic **repulsion** in the *syn* conformation [@Goodman2001]. This is an idea that has been floated since the 1970’s [@Lowe1973] but it gained new life with the Nature publication. People **disagreed** and the fight was on [@Bickelhaupt2004]. There are ongoing efforts to further refine the **model** for the energy difference [@Baranac2015; @Mo2011].

In [11]:
barrier = table.loc["ΔE (kJ/mol)", "syn conformer"]
literature = 12.1                                  # kJ/mol, Hirota et al. (1979)
print(f"HF/6-311G(d,p)++ barrier: {barrier:.1f} kJ/mol")
print(f"Literature barrier:       {literature:.1f} kJ/mol")
print(f"Difference:               {barrier - literature:+.1f} kJ/mol "
      f"({100 * (barrier - literature) / literature:+.0f} %)")

HF/6-311G(d,p)++ barrier: 12.8 kJ/mol
Literature barrier:       12.1 kJ/mol
Difference:               +0.7 kJ/mol (+5 %)


## Summary

We have learned more about **constructing** a *Z-matrix* and applied this knowledge to design a *Z-matrix* for ethane within which we can easily create any **torsion angle** we wish. We have observed how defining symmetry can **enforce** a given conformation while optimizing a structure, and that the unique coordinates must match the declared point group. We have let *Python* carry the coordinates from one level of theory to the next and collect the results in a table. We have observed that the structures and the energy difference were reasonably close to **literature values**.

In the exercise, you will also explore what happens when we remove the symmetry declaration. Go to the instructions for the exercise and have fun.

## Exercise

Read the accompanying discussion above and then try the following activities.

### The Activities

**Repeat** the activities described in the discussion but edit the input files to **add** a command to output data for **localized** orbitals. After you have result files for the optimized *D<sub>3h</sub>* and *D<sub>3d</sub>* structures, produce **images** of the molecules that display bond lengths and angles. Then produce images showing graphic representations of the unique **local bonding orbitals** (one C–C &sigma;-bond and one C–H &sigma;-bond will suffice). Report the energy difference for the two symmetries (*syn* and *anti*). Then present images for the bonding MOs for both symmetries. **Compare** their energies and their symmetries and discuss which best represent which local bonds (or combinations thereof.)

Now **set the symmetry** of the *syn* structure to *C<sub>1</sub>* (no symmetry – this will allow free bond rotation.) Run the optimization at the AM1 level. Is the result *syn* or *anti*? **Discuss** the result. If the result remains *syn* **propose** why *GAMESS* could not bring itself to start the bond rotation. Now **change** the torsion angle by 0.1&deg;. Run the optimization **again**. Discuss the result. **Compare** the energies of the *C<sub>1</sub>* *anti* and the *C<sub>1</sub>* *syn* structures at the HF/6-311G(d,p)++ level. **Compare** bond lengths and angles as well. Do they match with the results that used symmetry?

````{tip}
The `replacements` option of `gt.new_input()` edits an input file for you. For example, this writes a *C<sub>1</sub>* version of the *syn* AM1 input. With *C<sub>1</sub>* there must be **no blank line** after the symmetry line, so the blank line is replaced too.

```python
syn_c1_inp = gt.new_input(
    INPUTS / "Ethane_syn_AM1_Z.inp", RUNS / "Ethane_syn_AM1_Z_C1.inp",
    replacements={"DNH 3\n\n": "C1\n"})
syn_c1_log = gt.run_gamess(syn_c1_inp)
```

To change the torsion, replace the text of line 16 in the same way. `gt.internal_coordinates()` and `ethane_summary()` from this notebook will report the geometry, and `gt.molecular_orbitals()` and `gt.localized_fock()` (see [chapter 3](3_Visual_Input.ipynb)) the orbital energies. To time a job, put `%%time` on the first line of the cell that runs it.
````

### The Report

Create a ONE-PAGE **document** that presents the images of the molecules and local and molecular orbitals described above. **Discuss** the relative energies of the orbitals. Are there any degenerate sets? How many nodal planes are in each set? Can you see how the local orbitals could be combined to make the molecular orbitals?

Did fixing the **symmetry** also fix the bond angles in the *syn* configuration? **Present** the consequences of removing the symmetry constraint. If there was no change in torsion when the symmetry was removed, **propose** a reason why. Why did a slight tip in the torsion angle **solve** the problem? Present the bond lengths and angles for the HF/6-311G(d,p)++ and discuss how they relate to the values with symmetry.

Try **running** energy calculations at the HF/6-311G(d,p)++ level with and without symmetry and see if there is a difference in the **time** each calculation takes. Discuss the difference in time (there may be none as these are small molecules and modern personal computers are enormously powerful).

Include an **appendix** with the text for all the input files that you used (please do not include any result files – there will be no trees left!) For these files preface each with the file name and a note on its purpose and use a fixed-width font.

#### Appendix: Input files

As an example, this cell prints every input file used on this page, each with its file name.

In [12]:
files = [INPUTS / "Ethane_anti_AM1_Z.inp", RUNS / "Ethane_anti_6311_U.inp",
         INPUTS / "Ethane_syn_AM1_Z.inp", RUNS / "Ethane_syn_6311_U.inp"]
for f in files:
    print(f"===== {f} =====")
    print(f.read_text())

===== inputs/Ethane_anti_AM1_Z.inp =====
 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 
  COORD=ZMT $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=AM1 $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=100 $END
 $DATA 
anti ethane
DND 3

C  
C   1  1.5
H   2  1.0  1  110
H   2  1.0  1  110  3  120
H   2  1.0  1  110  3  -120
H   1  1.0  2  110  3  180
H   1  1.0  2  110  6  120
H   1  1.0  2  110  6  -120
 $END

===== runs/Ethane_anti_6311_U.inp =====
 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 
  COORD=UNIQUE NZVAR=18 $END
 $SYSTEM TIMLIM=525600 MEMORY=1000000 $END
 $BASIS GBASIS=N311 NGAUSS=6 NDFUNC=1 NPFUNC=1 DIFFSP=.TRUE. DIFFS=.TRUE. $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT OPTTOL=0.0001 NSTEP=20 $END
 $DATA 
anti ethane
DND 3

 C        6.0    0.0000000000    0.0000000000   -0.7501162007
 H        1.0    0.0000000000    1.0448521625    1.1453263085
 $END
 $ZMAT IZMAT(1)=1,1,2,
                1,8,1,  2,8,1,2,
                1

## AI Statement

This page was converted from the original Markdown page `4_Ethane.md` to a *Jupyter* notebook with the help of Claude (Anthropic), following the author's instructions.

- **Original text kept (about 85 %).** The introduction, learning goals, point-group and *Z-matrix* explanations, the *anti* and *syn* discussions, the note on the edited `IZMAT`, the controversy section with its references, the summary and the exercise are the author's words, as are the four input files. Small edits: typos fixed, the torsion is now called 6,1,2,3 to match the *Z-matrix* (the original said 6,1,2,8), the title of the *syn* AM1 input now reads "syn ethane", and the *syn* mirror plane is called *&sigma;<sub>v</sub>*.
- **Written by AI.** The settings cell, all code cells and the paragraphs that introduce them; the explanation of the `ENCODED Z MATRIX` table; the deliberate failed run with the wrong-plane coordinates and its explanation; the `ethane_summary()` function; the comparison with the literature barrier; the exercise tip; and the appendix cell.
- **Changed workflow.** Jobs are run with `run_gamess()`; HF/6-311G(d,p)++ input files are written by *Python* from the AM1 results instead of by copy and paste; the result tables are computed from the log files rather than typed (the numbers agree with the original to the digits shown).